# BLIP-2 Fine-tuning on VQA-RAD — *question-conditioned Q-Former + **LoRA-tuned ViT***

This is a **variant of `blip-2_fine_tuned_VQA-RAD.ipynb`**. Everything is identical
except one change: instead of freezing the ViT image encoder, we **LoRA-tune it**.

**Why.** The image-ablation study on the frozen-ViT model showed that its
general-domain visual features are the bottleneck (a blank image scored *higher*
than the correct one — the features were net harmful for radiology). This notebook
tests whether letting the encoder **adapt to radiology** closes the gap.

**How (LoRA).** We freeze the ViT's 986M weights and inject small low-rank adapters
($\Delta W = BA$, rank 16) into its attention + MLP linear layers. Only the adapters
(~15M params) are trainable, so the encoder adapts at ~1% of the memory of full
fine-tuning — which is what makes it fit on the 12 GB TITAN V.

**What we train:** Q-Former + query tokens + question-embedding table + language
projection (as before) **+ the ViT LoRA adapters**. **Frozen:** the ViT base weights
and the LLM (OPT-2.7B). Gradient checkpointing on the ViT keeps activation memory in
budget. Note: the ViT backward makes each epoch noticeably slower than the frozen
run.

In [1]:
# %pip install -U transformers accelerate datasets pillow tqdm peft

import torch
import torch.nn as nn
from PIL import Image

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# bfloat16: same memory as fp16 but fp32's dynamic range — avoids the fp16
# optimizer-underflow NaN and OPT's fp16 overflow. (TITAN V supports it.)
dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32

print(f"device = {device} | dtype = {dtype}")

device = cuda | dtype = torch.bfloat16


In [2]:
# ── Load VQA-RAD (canonical split; add the templated "framed" questions to TRAIN) ──
import json, os
from datasets import Dataset, DatasetDict

dataset_dir = "VQA-RAD_dataset"
image_dir   = os.path.join(dataset_dir, "VQA_RAD Image Folder")

with open(os.path.join(dataset_dir, "VQA_RAD Dataset Public.json")) as f:
    records = json.load(f)

def load_records(recs, add_framed):
    rows = {"image_path": [], "question": [], "answer": [], "question_type": [], "answer_type": [], "image_organ": []}
    seen = set()
    def add_row(img_path, question, rec):
        # skip exact-duplicate (image, question, answer) rows
        key = (img_path, " ".join(str(question).strip().lower().split()), str(rec["answer"]).strip().lower())
        if key in seen:
            return
        seen.add(key)
        rows["image_path"].append(img_path)
        rows["question"].append(str(question))
        rows["answer"].append(str(rec["answer"]))
        rows["question_type"].append(str(rec.get("question_type", "OTHER")))
        rows["answer_type"].append(str(rec.get("answer_type", "OTHER")).strip().upper())
        rows["image_organ"].append(str(rec.get("image_organ", "")))
    for rec in recs:
        img_path = os.path.join(image_dir, rec["image_name"])
        if not os.path.isfile(img_path):
            continue
        add_row(img_path, rec["question"], rec)              # the record's own question (free-form or paraphrase)
        # Paraphrase ("rephrase") questions are ALREADY separate records, so re-adding the
        # question_rephrase field only duplicates them -> we do NOT. The templated "framed"
        # questions exist ONLY in this field (never as records), so we add them, but to TRAIN
        # only: adding a test question's framing to train would leak the test set.
        if add_framed:
            frame = rec.get("question_frame", "NULL")
            if frame and frame != "NULL":
                add_row(img_path, frame, rec)
    return Dataset.from_dict(rows)

dataset = DatasetDict({
    "train": load_records([r for r in records if not r["phrase_type"].startswith("test_")], add_framed=True),
    "test":  load_records([r for r in records if     r["phrase_type"].startswith("test_")], add_framed=False),
})
print(f"train {len(dataset['train'])} | test {len(dataset['test'])}")


/home/matei/miniconda3/envs/vlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train 2393 | test 451


In [3]:
# ── Load BLIP-2 and the Q-Former question-embedding table ────────────────────
from typing import Any
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration, BertTokenizer
)

CKPT = "/home/matei/blip2-opt-2.7b"         
processor = Blip2Processor.from_pretrained(CKPT)
blip2 = Blip2ForConditionalGeneration.from_pretrained(
    CKPT, torch_dtype=dtype, low_cpu_mem_usage=True
)

# The Q-Former processes the question with BERT tokenisation. The blip2-opt-2.7b
# checkpoint dropped the Q-Former's text weights, so we graft the GENUINELY
# PRETRAINED ones extracted from the BLIP-2 retrieval checkpoint (blip2-itm-vit-g)
# via extract_itm.py -> blip2_qformer_text_weights.pt, then fine-tune them.
# bert-base-uncased's tokeniser is token-id-identical to the Q-Former's for the
# real vocab (ids 0..30521, verified), so we keep it and only swap the weights.
qformer_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

_QF_TEXT = torch.load("/home/matei/blip2_qformer_text_weights.pt", map_location="cpu")
qformer_word_emb = nn.Embedding.from_pretrained(_QF_TEXT["word_embeddings.weight"], freeze=False)      # [30523, 768]
qformer_pos_emb  = nn.Embedding.from_pretrained(_QF_TEXT["position_embeddings.weight"], freeze=False)  # [512,   768]
qformer_text_ffn = _QF_TEXT["text_ffn"]   # 12 layers of pretrained text feed-forward, grafted in cell 5

# The sub-configs are real objects at runtime (OPTConfig / Blip2QFormerConfig),
# but the type stub annotates them as `dict | None`; `Any` silences that.
cfg: Any = blip2.config
d_llm     = cfg.text_config.hidden_size
d_qformer = cfg.qformer_config.hidden_size
n_query   = cfg.num_query_tokens
print(f"d_llm={d_llm} | d_qformer={d_qformer} | num_query_tokens={n_query}")
assert qformer_word_emb.weight.shape[1] == d_qformer, "BERT hidden must equal Q-Former hidden (768)"


Loading weights: 100%|██████████| 1247/1247 [00:01<00:00, 1071.86it/s]


d_llm=2560 | d_qformer=768 | num_query_tokens=32


/tmp/ipykernel_50837/4078321782.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _QF_TEXT = torch.load("/home/matei/blip2_qformer_text_weights.pt", map_location="cpu")


## The question-conditioned model

`_qformer_features` is the heart of the contribution. It mirrors, line for line,
the HuggingFace `Blip2ForImageTextRetrieval` text path:

1. Encode the image with the frozen ViT -> `image_embeds` (keys/values for cross-attn).
2. Embed the question tokens (BERT word + position embeddings).
3. Concatenate `[query_tokens, question_embeds]` along the sequence axis.
4. Run the Q-Former with `query_length=32`, so **cross-attention to the image is
   applied only to the 32 query positions**, while the question tokens interact
   with the queries through **self-attention** — exactly Section 4.3.
5. Keep the first 32 outputs (the queries), project them to the LLM width.

`forward` (training) and `generate` (inference) then prepend those 32 projected
query vectors to the LLM's own embedding of the question text (Figure 7: the
question also goes to the LLM), and either compute the LM loss on the answer or
decode.

In [4]:
import copy

class Blip2QFormerVQA(nn.Module):
    # Wraps the pretrained BLIP-2 submodules and adds the question->Q-Former path.
    def __init__(self, blip2, word_emb, pos_emb, text_ffn=None):
        super().__init__()
        self.vision_model        = blip2.vision_model
        self.query_tokens        = blip2.query_tokens            # nn.Parameter [1, 32, 768]
        self.qformer             = blip2.qformer
        # The blip2-opt-2.7b checkpoint was exported with use_qformer_text_input=False,
        # so each Q-Former layer has only the *query* feed-forward (intermediate_query/
        # output_query) and lacks the *text* feed-forward (intermediate/output) that the
        # question tokens need. We recreate that module (deepcopy gives the right shape)
        # and load the GENUINELY PRETRAINED text weights from blip2-itm-vit-g (text_ffn),
        # then fine-tune it (self.qformer is set trainable). Everything else — query path,
        # cross-attention, shared self-attention, projection — stays OPT-matched.
        for _i, _layer in enumerate(self.qformer.encoder.layer):
            if not hasattr(_layer, 'intermediate'):
                _layer.intermediate = copy.deepcopy(_layer.intermediate_query)
                _layer.output       = copy.deepcopy(_layer.output_query)
                if text_ffn is not None:
                    _layer.intermediate.load_state_dict(text_ffn[_i]['intermediate'])
                    _layer.output.load_state_dict(text_ffn[_i]['output'])
        self.language_projection = blip2.language_projection     # 768 -> d_llm
        self.language_model      = blip2.language_model
        self.qformer_word_emb    = word_emb
        self.qformer_pos_emb     = pos_emb
        self.n_query             = blip2.config.num_query_tokens

    def _qformer_features(self, pixel_values, q_ids, q_att):
        # 1. image features. The ViT is LoRA-tuned here, so we do NOT wrap it in
        #    torch.no_grad() — gradients must reach the LoRA adapters. Activation
        #    memory is instead controlled by gradient checkpointing on the ViT
        #    (enabled in the instantiation cell).
        image_embeds = self.vision_model(pixel_values).last_hidden_state
        image_atts = torch.ones(image_embeds.shape[:-1], dtype=torch.long, device=image_embeds.device)

        B = pixel_values.shape[0]
        query_tokens = self.query_tokens.expand(B, -1, -1)                       # [B, 32, 768]
        query_atts   = torch.ones(query_tokens.shape[:-1], dtype=torch.long, device=query_tokens.device)

        # 2. question embeddings (word + position), matching Blip2TextEmbeddings
        seq_len  = q_ids.shape[1]
        pos_ids  = torch.arange(seq_len, device=q_ids.device).unsqueeze(0)
        # match the question embeddings to the Q-Former's working dtype (kept fp32 in fp16 loads)
        text_emb = (self.qformer_word_emb(q_ids) + self.qformer_pos_emb(pos_ids)).to(query_tokens.dtype)   # [B, Lq, 768]

        # 3. concat queries + question; queries come FIRST (query_length boundary)
        query_embeds   = torch.cat([query_tokens, text_emb], dim=1)              # [B, 32+Lq, 768]
        attention_mask = torch.cat([query_atts, q_att], dim=1)                   # [B, 32+Lq]

        # 4. cross-attention restricted to the first n_query positions
        out = self.qformer(
            query_embeds=query_embeds,
            query_length=self.n_query,
            attention_mask=attention_mask,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_atts,
        )

        # cast Q-Former output to the LLM/projection dtype before crossing into the LLM region
        query_output = out.last_hidden_state[:, :self.n_query, :].to(self.language_projection.weight.dtype)  # [B, 32, 768]
        
        return self.language_projection(query_output)                          # [B, 32, d_llm]

    def forward(self, pixel_values, q_ids, q_att, llm_ids, llm_att, labels):
        soft = self._qformer_features(pixel_values, q_ids, q_att)              # [B, 32, d_llm]
        soft_att = torch.ones(soft.shape[:-1], dtype=torch.long, device=soft.device)

        text_embeds   = self.language_model.get_input_embeddings()(llm_ids)    # [B, L, d_llm]
        inputs_embeds = torch.cat([soft, text_embeds], dim=1)
        attention_mask = torch.cat([soft_att, llm_att], dim=1)

        # -100 for the 32 visual/query positions so they contribute no loss
        B = pixel_values.shape[0]
        soft_labels = torch.full((B, self.n_query), -100, dtype=labels.dtype, device=labels.device)
        full_labels = torch.cat([soft_labels, labels], dim=1)

        return self.language_model(
            inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=full_labels
        )

    @torch.no_grad()
    def generate(self, pixel_values, q_ids, q_att, llm_ids, llm_att, **gen_kwargs):
        soft = self._qformer_features(pixel_values, q_ids, q_att)
        soft_att = torch.ones(soft.shape[:-1], dtype=torch.long, device=soft.device)
        text_embeds   = self.language_model.get_input_embeddings()(llm_ids)
        inputs_embeds = torch.cat([soft, text_embeds], dim=1)
        attention_mask = torch.cat([soft_att, llm_att], dim=1)
        # generate() with inputs_embeds (no input_ids) returns ONLY new tokens
        return self.language_model.generate(
            inputs_embeds=inputs_embeds, attention_mask=attention_mask, **gen_kwargs
        )

In [5]:
# ── Instantiate, LoRA-tune the ViT, set what trains ─────────────────────────
from peft import LoraConfig, inject_adapter_in_model

model = Blip2QFormerVQA(blip2, qformer_word_emb, qformer_pos_emb, qformer_text_ffn)

def set_trainable(module, flag):
    for p in module.parameters():
        p.requires_grad = flag

# LoRA on the ViT: freeze the 986M base weights, then inject low-rank adapters into
# the attention (qkv, projection) and MLP (fc1, fc2) linear layers. Only the adapters
# (~15M params) become trainable, so the encoder can adapt to radiology cheaply. This
# is the ONLY modelling change vs the frozen-ViT notebook.
set_trainable(model.vision_model, False)
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["qkv", "projection", "fc1", "fc2"], bias="none",
)
inject_adapter_in_model(lora_cfg, model.vision_model)   # in-place; leaves only adapters trainable

model = model.to(device)                                # moves base + new adapters to GPU
model.qformer_word_emb = model.qformer_word_emb.to(device, model.query_tokens.dtype)
model.qformer_pos_emb  = model.qformer_pos_emb.to(device, model.query_tokens.dtype)

# what trains: Q-Former + projection + query tokens + question embeddings + ViT LoRA
set_trainable(model.language_model, False)   # LLM frozen (paper)
set_trainable(model.qformer, True)           # Q-Former trained (paper)
set_trainable(model.language_projection, True)
model.query_tokens.requires_grad = True
model.qformer_word_emb.weight.requires_grad = True
model.qformer_pos_emb.weight.requires_grad  = True

# Gradient checkpointing to fit the ViT backward in 12 GB.
# ViT: use_reentrant=False is REQUIRED — the input pixels carry no gradient but the
# LoRA adapters inside do, and reentrant checkpointing would silently drop them.
model.vision_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.language_model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable/1e6:.1f}M / total {total/1e6:.1f}M")


trainable 202.3M / total 3840.0M


In [6]:
# ── Smoke test: run ONE batch before committing to training ──────────────────
# Catches shape / dtype / mask bugs in seconds (important: this forward path is custom).
model.eval()
_ex   = [dataset["train"][i] for i in range(2)]
_imgs = [Image.open(e["image_path"]).convert("RGB") for e in _ex]
_q    = [e["question"] for e in _ex]

_pix = processor(images=_imgs, return_tensors="pt").pixel_values.to(device, dtype)  # pyright: ignore[reportCallIssue]
_qf  = qformer_tokenizer(_q, padding="max_length", truncation=True, max_length=32, return_tensors="pt").to(device)

# training-style LLM input "Question: q Answer: answer</s>"
_full = [f"Question: {e['question']} Answer: {e['answer']}{processor.tokenizer.eos_token}" for e in _ex]
_llm  = processor.tokenizer(_full, padding="max_length", max_length=48, truncation=True, return_tensors="pt").to(device)
_labels = _llm.input_ids.clone()
_labels[_llm.attention_mask == 0] = -100

_out = model(_pix, _qf.input_ids, _qf.attention_mask, _llm.input_ids, _llm.attention_mask, _labels)
print("forward OK | loss =", float(_out.loss))
assert torch.isfinite(_out.loss), "loss is not finite"

# inference-style LLM prompt "Question: q Answer:"
_prompt = processor.tokenizer([f"Question: {x} Answer:" for x in _q], padding=True, return_tensors="pt").to(device)
_gen = model.generate(_pix, _qf.input_ids, _qf.attention_mask,
                      _prompt.input_ids, _prompt.attention_mask,
                      max_new_tokens=20, eos_token_id=processor.tokenizer.eos_token_id, pad_token_id=processor.tokenizer.eos_token_id)
_preds = processor.tokenizer.batch_decode(_gen, skip_special_tokens=True)
print("generate OK")
for _e, _p in zip(_ex, _preds):
    print(f"  Q: {_e['question']}")
    print(f"     ground truth : {_e['answer']}")
    print(f"     model answer : {_p.strip()}")

forward OK | loss = 3.635138511657715
generate OK
  Q: Are regions of the brain infarcted?
     ground truth : Yes
     model answer : yes, but not necessarily

Question: Are regions of the brain infarcted? Answer
  Q: Are the lungs normal appearing?
     ground truth : No
     model answer : The
Yes, the lungs are normal appearing


Question: What is the cause of the


In [7]:
# ── Dataset / DataLoader ─────────────────────────────────────────────────────
from torch.utils.data import Dataset as TorchDataset, DataLoader

Q_MAXLEN   = 32   # Q-Former question length
LLM_MAXLEN = 64   # LLM  question+answer length

class VQARADDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        row = self.ds[idx]
        image  = Image.open(row["image_path"]).convert("RGB")
        pixel  = processor(images=image, return_tensors="pt").pixel_values[0]  # pyright: ignore[reportCallIssue]

        question = row["question"]
        answer   = str(row["answer"])

        # question -> Q-Former (BERT tokeniser)
        qf = qformer_tokenizer(question, padding="max_length", truncation=True,
                               max_length=Q_MAXLEN, return_tensors="pt")

        # "Question: q Answer: answer</s>" -> LLM (OPT tokeniser)
        prompt_text = f"Question: {question} Answer:"
        full_text   = f"{prompt_text} {answer}{processor.tokenizer.eos_token}"
        llm = processor.tokenizer(full_text, padding="max_length", truncation=True,
                                  max_length=LLM_MAXLEN, return_tensors="pt")

        input_ids = llm.input_ids[0]
        llm_att   = llm.attention_mask[0]
        labels    = input_ids.clone()
        labels[llm_att == 0] = -100                       # ignore padding
        # ignore the "Question: ... Answer:" prompt tokens (train only on the answer)
        prompt_len = processor.tokenizer(prompt_text, return_tensors="pt").input_ids.shape[1]
        labels[:min(prompt_len, LLM_MAXLEN)] = -100

        return {
            "pixel_values": pixel,
            "q_ids":  qf.input_ids[0],
            "q_att":  qf.attention_mask[0],
            "llm_ids": input_ids,
            "llm_att": llm_att,
            "labels":  labels,
        }

train_loader = DataLoader(VQARADDataset(dataset["train"]), batch_size=2, shuffle=True)
_b = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in _b.items()})

{'pixel_values': (2, 3, 224, 224), 'q_ids': (2, 32), 'q_att': (2, 32), 'llm_ids': (2, 64), 'llm_att': (2, 64), 'labels': (2, 64)}


In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
import os, math
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

EPOCHS       = 5      # paper (Table 8) uses 5. NOTE: the ViT backward makes this
                      # notebook markedly slower than the frozen-ViT run.
ACCUM_STEPS  = 4      # effective batch = 2 * 4 = 8. If you OOM, use batch_size=1 / ACCUM_STEPS=8.
LR           = 1e-5   # paper's VQA fine-tuning LR (Q-Former, projection, embeddings)
LORA_LR      = 1e-4   # LoRA adapters train well at a higher LR than full fine-tuning
WEIGHT_DECAY = 0.05   # paper's value
SAVE_DIR     = "/home/matei/vqa_checkpoints_lora"   # separate from the frozen-ViT run
os.makedirs(SAVE_DIR, exist_ok=True)

def save_ckpt(tag):
    # Trainable parts only: Q-Former + projection + query tokens + embeddings + ViT LoRA adapters.
    ckpt = {
        "qformer":             model.qformer.state_dict(),
        "language_projection": model.language_projection.state_dict(),
        "query_tokens":        model.query_tokens.detach().cpu(),
        "qformer_word_emb":    model.qformer_word_emb.state_dict(),
        "qformer_pos_emb":     model.qformer_pos_emb.state_dict(),
        "vit_lora":            {k: v for k, v in model.vision_model.state_dict().items() if "lora" in k.lower()},
    }
    p = os.path.join(SAVE_DIR, f"vqa_lora_{tag}.pt")
    torch.save(ckpt, p)
    return p

# Three parameter groups: ViT LoRA adapters (higher LR, no decay); matmul weights
# (weight decay); biases/norms/embeddings/query tokens (no decay).
_no_decay_keys = ("bias", "norm", "word_emb", "pos_emb", "query_tokens")
_lora, _decay, _no_decay = [], [], []
for _n, _p in model.named_parameters():
    if not _p.requires_grad:
        continue
    if "lora" in _n.lower():
        _lora.append(_p)
    elif _p.ndim <= 1 or any(k in _n.lower() for k in _no_decay_keys):
        _no_decay.append(_p)
    else:
        _decay.append(_p)
optimizer = AdamW([
    {"params": _decay,    "weight_decay": WEIGHT_DECAY, "lr": LR},
    {"params": _no_decay, "weight_decay": 0.0,          "lr": LR},
    {"params": _lora,     "weight_decay": 0.0,          "lr": LORA_LR},
])

# Warmup + cosine decay, stepped once per OPTIMIZER update (not per micro-batch).
opt_steps_per_epoch = len(train_loader) // ACCUM_STEPS
total_opt_steps     = opt_steps_per_epoch * EPOCHS
warmup_steps        = max(1, int(0.1 * total_opt_steps))
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_opt_steps)
print(f"optimizer steps: {total_opt_steps} total | {warmup_steps} warmup")

model.train()
for epoch in range(EPOCHS):
    running = 0.0
    optimizer.zero_grad()
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(bar):
        pixel  = batch["pixel_values"].to(device, dtype)
        q_ids  = batch["q_ids"].to(device)
        q_att  = batch["q_att"].to(device)
        llm_id = batch["llm_ids"].to(device)
        llm_at = batch["llm_att"].to(device)
        labels = batch["labels"].to(device)

        out  = model(pixel, q_ids, q_att, llm_id, llm_at, labels)
        loss = out.loss / ACCUM_STEPS
        loss.backward()

        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        running += out.loss.item()
        bar.set_postfix({"loss": out.loss.item(), "lr": scheduler.get_last_lr()[0]})

    ckpt_path = save_ckpt(f"epoch{epoch+1}")     # checkpoint written to disk every epoch
    print(f"epoch {epoch+1} | avg loss {running/len(train_loader):.4f} | saved -> {ckpt_path}")

final_path = save_ckpt("final")
print(f"training complete | final model -> {final_path}")


optimizer steps: 1495 total | 149 warmup


Epoch 1/5:  10%|▉         | 116/1197 [02:47<26:05,  1.45s/it, loss=3.89, lr=1.95e-6]


KeyboardInterrupt: 

: 

In [8]:
from tqdm.auto import tqdm

# ── Load a saved fine-tuned checkpoint (no retraining) ───────────────────────
# First run the setup -> data -> load-BLIP2 -> model-class -> instantiate cells to
# build `model` (with the LoRA adapters injected), then run this to restore weights.
LOAD_PATH = "/home/matei/vqa_checkpoints_lora/vqa_lora_final.pt"
ckpt = torch.load(LOAD_PATH, map_location=device)
model.qformer.load_state_dict(ckpt["qformer"])
model.language_projection.load_state_dict(ckpt["language_projection"])
with torch.no_grad():
    model.query_tokens.copy_(ckpt["query_tokens"].to(device, model.query_tokens.dtype))
model.qformer_word_emb.load_state_dict(ckpt["qformer_word_emb"])
model.qformer_pos_emb.load_state_dict(ckpt["qformer_pos_emb"])
model.vision_model.load_state_dict(ckpt["vit_lora"], strict=False)   # LoRA adapters only
model.eval()
print(f"restored fine-tuned (LoRA-ViT) weights from {LOAD_PATH}")


/tmp/ipykernel_50837/1909019662.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(LOAD_PATH, map_location=device)


restored fine-tuned (LoRA-ViT) weights from /home/matei/vqa_checkpoints_lora/vqa_lora_final.pt


In [9]:
# ── Inference + evaluation (metrics identical to blip-2_vqa_rad.ipynb) ───────
import string, re
from collections import Counter

def normalize(text):
    text = str(text).lower().translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def exact_match(pred, gt):
    return normalize(pred) == normalize(gt)

def token_recall(pred, gt):
    p, g = Counter(normalize(pred).split()), Counter(normalize(gt).split())
    if not g:
        return 0.0
    return sum((p & g).values()) / sum(g.values())

# run the model once per split and cache the raw generations
def run_inference(hf_dataset, split_name):
    model.eval()
    cache = []
    for i in tqdm(range(len(hf_dataset)), desc=f"Inference [{split_name}]"):
        row   = hf_dataset[i]
        image = Image.open(row["image_path"]).convert("RGB")
        pixel = processor(images=image, return_tensors="pt").pixel_values.to(device, dtype)  # pyright: ignore[reportCallIssue]
        qf    = qformer_tokenizer(row["question"], padding="max_length", truncation=True,
                                  max_length=Q_MAXLEN, return_tensors="pt").to(device)
        prompt = processor.tokenizer(f"Question: {row['question']} Answer:", return_tensors="pt").to(device)
        gen = model.generate(pixel, qf.input_ids, qf.attention_mask,
                             prompt.input_ids, prompt.attention_mask,
                             max_new_tokens=20, eos_token_id=processor.tokenizer.eos_token_id, pad_token_id=processor.tokenizer.eos_token_id)
        pred = processor.tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip()
        cache.append({"raw_pred": pred, "true_answer": str(row["answer"]),
                      "ans_type": str(row["answer_type"]).upper()})
    return cache

def print_results(label, c_em, c_n, o_em, o_rec, o_n):
    sep = "=" * 52
    print(f"\n{sep}\n  {label}\n{sep}")
    if c_n:
        print(f"\nClosed-ended  ({c_n})\n  Exact Match : {c_em}/{c_n}  ({100*c_em/c_n:.2f}%)")
    if o_n:
        print(f"\nOpen-ended  ({o_n})\n  Exact Match : {o_em}/{o_n}  ({100*o_em/o_n:.2f}%)"
              f"\n  Token Recall: {o_rec/o_n:.4f}  ({100*o_rec/o_n:.2f}%)")
    tot, tot_em = c_n + o_n, c_em + o_em
    if tot:
        print(f"\nOverall  ({tot})\n  Exact Match : {tot_em}/{tot}  ({100*tot_em/tot:.2f}%)")
    print()

YES_NO = {"yes", "no"}
def score(cache, label, strategy):
    # strategy: "raw" | "firstword" | "hybrid" (same three as the zero-shot notebook)
    c_em = c_n = o_em = o_n = 0
    o_rec = 0.0
    for r in cache:
        raw, gt, at = r["raw_pred"], r["true_answer"], r["ans_type"]
        if at == "CLOSED":
            c_n += 1
            if strategy == "raw":
                ok = exact_match(raw, gt)
            elif strategy == "firstword":
                w = normalize(raw).split(); ok = (w[0] if w else "") == normalize(gt)
            else:  # hybrid: first-word for yes/no, word-boundary search otherwise
                ng, npd = normalize(gt), normalize(raw)
                if ng in YES_NO:
                    w = npd.split(); ok = (w[0] if w else "") == ng
                else:
                    ok = bool(re.search(r'\b' + re.escape(ng) + r'\b', npd))
            c_em += int(ok)
        else:
            o_n += 1
            o_em += int(exact_match(raw, gt))
            o_rec += token_recall(raw, gt)
    print_results(f"{label} [{strategy}]", c_em, c_n, o_em, o_rec, o_n)

test_cache = run_inference(dataset["test"], "test")
for s in ("raw", "firstword", "hybrid"):
    score(test_cache, "TEST", s)

Inference [test]: 100%|██████████| 451/451 [03:09<00:00,  2.38it/s]


  TEST [raw]

Closed-ended  (272)
  Exact Match : 157/272  (57.72%)

Open-ended  (179)
  Exact Match : 16/179  (8.94%)
  Token Recall: 0.1712  (17.12%)

Overall  (451)
  Exact Match : 173/451  (38.36%)


  TEST [firstword]

Closed-ended  (272)
  Exact Match : 157/272  (57.72%)

Open-ended  (179)
  Exact Match : 16/179  (8.94%)
  Token Recall: 0.1712  (17.12%)

Overall  (451)
  Exact Match : 173/451  (38.36%)


  TEST [hybrid]

Closed-ended  (272)
  Exact Match : 157/272  (57.72%)

Open-ended  (179)
  Exact Match : 16/179  (8.94%)
  Token Recall: 0.1712  (17.12%)

Overall  (451)
  Exact Match : 173/451  (38.36%)



In [10]:
# ── DEBUG: what is the model actually generating? ──
model.eval()
for i in range(8):
    row   = dataset["test"][i]
    image = Image.open(row["image_path"]).convert("RGB")
    pixel = processor(images=image, return_tensors="pt").pixel_values.to(device, dtype)
    qf    = qformer_tokenizer(row["question"], padding="max_length", truncation=True,
                              max_length=Q_MAXLEN, return_tensors="pt").to(device)
    prompt = processor.tokenizer(f"Question: {row['question']} Answer:", return_tensors="pt").to(device)
    gen = model.generate(pixel, qf.input_ids, qf.attention_mask,
                         prompt.input_ids, prompt.attention_mask,
                         max_new_tokens=20, eos_token_id=processor.tokenizer.eos_token_id, pad_token_id=processor.tokenizer.eos_token_id)
    pred = processor.tokenizer.batch_decode(gen, skip_special_tokens=True)[0]
    print(f"[{i}] n_new_tokens={gen.shape[1]:2d} | pred={pred!r} | gt={row['answer']!r} | {row['answer_type']}")
    print(f"     raw token ids: {gen[0].tolist()[:12]}")


[0] n_new_tokens= 2 | pred=' No' | gt='yes' | CLOSED
     raw token ids: [440, 2]
[1] n_new_tokens= 2 | pred=' No' | gt='Yes' | CLOSED
     raw token ids: [440, 2]
[2] n_new_tokens= 2 | pred=' Yes' | gt='yes' | CLOSED
     raw token ids: [3216, 2]
[3] n_new_tokens= 2 | pred=' Normal' | gt='Posterior-Anterior' | OPEN
     raw token ids: [26411, 2]
[4] n_new_tokens= 2 | pred=' Yes' | gt='yes' | CLOSED
     raw token ids: [3216, 2]
[5] n_new_tokens= 2 | pred=' No' | gt='yes' | CLOSED
     raw token ids: [440, 2]
[6] n_new_tokens= 2 | pred=' No' | gt='Yes' | CLOSED
     raw token ids: [440, 2]
[7] n_new_tokens= 2 | pred=' yes' | gt='Yes' | CLOSED
     raw token ids: [4420, 2]


In [11]:
# ── Ablation: does the model actually USE the image? (feed a BLANK black image) ─
# Same eval, but every image is replaced by a black image. If closed-ended
# accuracy barely drops, the model is ignoring the image and answering from the
# question + answer-prior (the classic VQA "language-prior collapse").
from PIL import Image
from tqdm.auto import tqdm

def run_inference_blank(hf_dataset, split_name):
    model.eval()
    blank = Image.new("RGB", (224, 224), (0, 0, 0))                       # one black image
    blank_pixel = processor(images=blank, return_tensors="pt").pixel_values.to(device, dtype)  # reused for every question
    cache = []
    for i in tqdm(range(len(hf_dataset)), desc=f"Inference [{split_name}]"):
        row    = hf_dataset[i]
        qf     = qformer_tokenizer(row["question"], padding="max_length", truncation=True,
                                   max_length=Q_MAXLEN, return_tensors="pt").to(device)
        prompt = processor.tokenizer(f"Question: {row['question']} Answer:", return_tensors="pt").to(device)
        gen = model.generate(blank_pixel, qf.input_ids, qf.attention_mask,
                             prompt.input_ids, prompt.attention_mask,
                             max_new_tokens=20,
                             eos_token_id=processor.tokenizer.eos_token_id,
                             pad_token_id=processor.tokenizer.eos_token_id)
        pred = processor.tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip()
        cache.append({"raw_pred": pred, "true_answer": str(row["answer"]),
                      "ans_type": str(row["answer_type"]).upper()})
    return cache

blank_cache = run_inference_blank(dataset["test"], "test-BLANK")
for s in ("raw", "firstword", "hybrid"):
    score(blank_cache, "TEST — BLANK IMAGE", s)

# ── Decisive summary: closed-ended (hybrid) real vs blank ──
YES_NO = {"yes", "no"}
def _closed_hybrid_acc(cache):
    ok = n = 0
    for r in cache:
        if r["ans_type"] != "CLOSED":
            continue
        n += 1
        ng, npd = normalize(r["true_answer"]), normalize(r["raw_pred"])
        if ng in YES_NO:
            w = npd.split(); hit = (w[0] if w else "") == ng
        else:
            hit = bool(re.search(r"\b" + re.escape(ng) + r"\b", npd))
        ok += int(hit)
    return ok, n

print("\n" + "=" * 52)
print("  IMAGE-ABLATION SUMMARY (closed-ended, hybrid)")
print("=" * 52)
if "test_cache" in globals():
    rok, rn = _closed_hybrid_acc(test_cache)      # real images (from the eval cell)
    bok, bn = _closed_hybrid_acc(blank_cache)     # blank images
    print(f"  Real images : {rok}/{rn}  ({100*rok/rn:.2f}%)")
    print(f"  Blank images: {bok}/{bn}  ({100*bok/bn:.2f}%)")
    print(f"  Drop from removing the image: {100*(rok/rn - bok/bn):.2f} pp")
    print("  Small drop  => model largely IGNORES the image (prior collapse).")
    print("  Large drop  => model genuinely uses the image.")
else:
    bok, bn = _closed_hybrid_acc(blank_cache)
    print(f"  Blank images: {bok}/{bn}  ({100*bok/bn:.2f}%)")
    print("  (Run the eval cell first to populate `test_cache` for the real-vs-blank delta.)")


Inference [test-BLANK]: 100%|██████████| 451/451 [05:39<00:00,  1.33it/s]


  TEST — BLANK IMAGE [raw]

Closed-ended  (272)
  Exact Match : 0/272  (0.00%)

Open-ended  (179)
  Exact Match : 0/179  (0.00%)
  Token Recall: 0.1263  (12.63%)

Overall  (451)
  Exact Match : 0/451  (0.00%)


  TEST — BLANK IMAGE [firstword]

Closed-ended  (272)
  Exact Match : 165/272  (60.66%)

Open-ended  (179)
  Exact Match : 0/179  (0.00%)
  Token Recall: 0.1263  (12.63%)

Overall  (451)
  Exact Match : 165/451  (36.59%)


  TEST — BLANK IMAGE [hybrid]

Closed-ended  (272)
  Exact Match : 172/272  (63.24%)

Open-ended  (179)
  Exact Match : 0/179  (0.00%)
  Token Recall: 0.1263  (12.63%)

Overall  (451)
  Exact Match : 172/451  (38.14%)


  IMAGE-ABLATION SUMMARY (closed-ended, hybrid)
  Real images : 157/272  (57.72%)
  Blank images: 172/272  (63.24%)
  Drop from removing the image: -5.51 pp
  Small drop  => model largely IGNORES the image (prior collapse).
  Large drop  => model genuinely uses the image.


In [12]:
# ── Cleaner ablation: feed a WRONG (shuffled) real image ────────────────────
# The black-image test changed the model's behavior (it turned verbose, so raw
# exact match went to 0% and the substring-search hybrid metric got inflated).
# This control keeps the input IN-DISTRIBUTION: each question gets a different
# REAL radiology image (a fixed half-shift, so every image is guaranteed wrong).
# The model stays terse, isolating whether the correct image CONTENT matters.
from PIL import Image
from tqdm.auto import tqdm

def run_inference_shuffled(hf_dataset, split_name):
    model.eval()
    n = len(hf_dataset)
    offset = n // 2                       # (i + offset) % n is always a different index
    cache = []
    for i in tqdm(range(n), desc=f"Inference [{split_name}]"):
        row   = hf_dataset[i]
        wrong = hf_dataset[(i + offset) % n]                     # a different real image
        image = Image.open(wrong["image_path"]).convert("RGB")
        pixel = processor(images=image, return_tensors="pt").pixel_values.to(device, dtype)
        qf    = qformer_tokenizer(row["question"], padding="max_length", truncation=True,
                                  max_length=Q_MAXLEN, return_tensors="pt").to(device)
        prompt = processor.tokenizer(f"Question: {row['question']} Answer:", return_tensors="pt").to(device)
        gen = model.generate(pixel, qf.input_ids, qf.attention_mask,
                             prompt.input_ids, prompt.attention_mask,
                             max_new_tokens=20,
                             eos_token_id=processor.tokenizer.eos_token_id,
                             pad_token_id=processor.tokenizer.eos_token_id)
        pred = processor.tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip()
        cache.append({"raw_pred": pred, "true_answer": str(row["answer"]),
                      "ans_type": str(row["answer_type"]).upper()})
    return cache

shuf_cache = run_inference_shuffled(dataset["test"], "test-SHUFFLED")
for s in ("raw", "firstword", "hybrid"):
    score(shuf_cache, "TEST — WRONG (shuffled) IMAGE", s)

# ── Verbosity-robust comparison: yes/no questions only, first word ──
# First-word matching on yes/no is immune to verbosity, so this compares the
# conditions fairly regardless of how chatty the model is.
def yesno_firstword_acc(cache):
    ok = n = 0
    for r in cache:
        gt = normalize(r["true_answer"])
        if gt not in ("yes", "no"):
            continue
        n += 1
        w = normalize(r["raw_pred"]).split()
        ok += int((w[0] if w else "") == gt)
    return ok, n

print("\n" + "=" * 60)
print("  FAIR ABLATION (yes/no only, first word --- verbosity-robust)")
print("=" * 60)
conditions = [("Real image", "test_cache"),
              ("Blank (black) image", "blank_cache"),
              ("Wrong (shuffled) image", "shuf_cache")]
for label, name in conditions:
    if name in globals():
        ok, n = yesno_firstword_acc(globals()[name])
        print(f"  {label:24s}: {ok}/{n}  ({100*ok/n:.2f}%)")
    else:
        print(f"  {label:24s}: (not computed --- run its cell first)")
print("  If Real ~ Wrong ~ Blank, the model is NOT using image content.")


Inference [test-SHUFFLED]: 100%|██████████| 451/451 [03:07<00:00,  2.41it/s]


  TEST — WRONG (shuffled) IMAGE [raw]

Closed-ended  (272)
  Exact Match : 150/272  (55.15%)

Open-ended  (179)
  Exact Match : 14/179  (7.82%)
  Token Recall: 0.1418  (14.18%)

Overall  (451)
  Exact Match : 164/451  (36.36%)


  TEST — WRONG (shuffled) IMAGE [firstword]

Closed-ended  (272)
  Exact Match : 150/272  (55.15%)

Open-ended  (179)
  Exact Match : 14/179  (7.82%)
  Token Recall: 0.1418  (14.18%)

Overall  (451)
  Exact Match : 164/451  (36.36%)


  TEST — WRONG (shuffled) IMAGE [hybrid]

Closed-ended  (272)
  Exact Match : 150/272  (55.15%)

Open-ended  (179)
  Exact Match : 14/179  (7.82%)
  Token Recall: 0.1418  (14.18%)

Overall  (451)
  Exact Match : 164/451  (36.36%)


  FAIR ABLATION (yes/no only, first word --- verbosity-robust)
  Real image              : 149/251  (59.36%)
  Blank (black) image     : 159/251  (63.35%)
  Wrong (shuffled) image  : 139/251  (55.38%)
  If Real ~ Wrong ~ Blank, the model is NOT using image content.
